In [1]:
import os, sys
os.chdir("/files/week-03/Project/code")
sys.path.insert(0, os.getcwd())
print(os.getcwd(), os.path.exists("helpers.py"))

/files/week-03/Project/code True


In [2]:
from helpers import *

train, test = load_returns()
noms = list(ACTIONS.keys())
print(train.shape, test.shape, A.shape)

(437, 5) (320, 5) (6, 4)


In [3]:
R_train = train[FACT].values

Q_reg = train_q(R_train, train["regime_lag"].values, n_states=4)
Q_noreg = train_q(R_train, np.zeros(len(train), dtype=int), n_states=1)

print("Politique avec régime :", {r: noms[a] for r, a in enumerate(Q_reg.argmax(axis=1))})
print("Politique sans régime :", noms[Q_noreg.argmax()])

res = {}
for nom, data in [("train", train), ("test", test)]:
    res[(nom, "poids egaux")] = perf(data[FACT].mean(axis=1))
    res[(nom, "Q-learning avec regime")] = perf(evaluer(Q_reg, data, True))
    res[(nom, "Q-learning sans regime")] = perf(evaluer(Q_noreg, data, False))

tableau = pd.DataFrame(res).T.round(3)
tableau.to_csv("../results/qlearning_performance.csv")
print(tableau)

Politique avec régime : {0: 'quality+momentum', 1: 'size+momentum', 2: 'momentum', 3: 'momentum'}
Politique sans régime : value+momentum
                              rendement_cumule  sharpe  max_drawdown
train poids egaux                        4.661   1.284        -0.104
      Q-learning avec regime            16.785   0.951        -0.161
      Q-learning sans regime            12.967   1.131        -0.112
test  poids egaux                        1.055   0.472        -0.172
      Q-learning avec regime             0.830   0.246        -0.365
      Q-learning sans regime             0.755   0.267        -0.407


In [4]:
res = []
for risk in [10, 50, 200]:
    for seed in range(10):
        Qr = train_q(R_train, train["regime_lag"].values, n_states=4, risk=risk, seed=seed)
        Qn = train_q(R_train, np.zeros(len(train), dtype=int), n_states=1, risk=risk, seed=seed)
        res.append({
            "risk": risk,
            "seed": seed,
            "sharpe_avec": perf(evaluer(Qr, test, True))["sharpe"],
            "sharpe_sans": perf(evaluer(Qn, test, False))["sharpe"],
        })

res = pd.DataFrame(res)
sens = res.groupby("risk")[["sharpe_avec", "sharpe_sans"]].agg(["mean", "std"]).round(3)
sens.to_csv("../results/sensibilite_risk.csv")
print(sens)

     sharpe_avec        sharpe_sans       
            mean    std        mean    std
risk                                      
10         0.307  0.097       0.299  0.139
50         0.449  0.037       0.464  0.026
200        0.456  0.034       0.451  0.065
